# Flagship CIC-IDS2017 model
I will use a day-by-day split to simulate real-world intrustion detection with zero-day (unseen) attacks. 

In [43]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
import glob
import re
import unicodedata
from sklearn.model_selection import train_test_split

RANDOM_STATE = 5

## Data Preperation
Greater detail / notes on data-prep in the imbalance experiment notebook.

In [44]:
files = glob.glob("data/*.csv")

day_re = "/([a-zA-Z]*)-"

def read_day_csv(f):
    df = pd.read_csv(f, encoding_errors="replace")
    df["day"] = re.search(day_re, f).group(1)
    return df

df = (
    pd.concat((read_day_csv(f) for f in files), ignore_index=True)
    .rename(columns=lambda s: s.strip())
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
)

# Safely downcasts 64-bit data to 32-bit where there is no information loss
def down_cast(d):
    original_size = d.memory_usage(deep=True).sum()
    for col in d.select_dtypes("integer").columns:
        d[col] = pd.to_numeric(d[col], downcast="integer")
    for col in d.select_dtypes("float").columns:
        d[col] = pd.to_numeric(d[col], downcast="float")
    final_size = d.memory_usage(deep=True).sum()
    return d
df = down_cast(df)

df = df.drop_duplicates()

def normalise(s):
    return (unicodedata.normalize("NFKD", str(s))
            .encode("ascii", "ignore")
            .decode("ascii"))
df["Label"] = df["Label"].apply(normalise)

train_days = df["day"].isin(["Monday", "Tuesday", "Wednesday"])
train = df[train_days]
test  = df[~train_days]

train_attacks = set(train["Label"]) - {"BENIGN"}
test_attacks  = set(test["Label"])  - {"BENIGN"}
novel         = test_attacks - train_attacks

X_train = train.drop(columns=["Label", "day"])
X_test  = test.drop(columns=["Label", "day"])

y_train_attackname = train["Label"]
y_test_attackname  = test["Label"]

multi_to_binary = lambda name: 0 if (name=="BENIGN") else 1
y_train = y_train_attackname.map(multi_to_binary)
y_test = y_test_attackname.map(multi_to_binary)

X_train, X_cv, y_train, y_cv = train_test_split(
    X_train, y_train, test_size=0.2, random_state=RANDOM_STATE)

# For XGBoost, don't need scaling
# from sklearn.preprocessing import StandardScaler
# std_scaler = StandardScaler().fit(X_train)
# X_train = std_scaler.transform(X_train)
# X_cv    = std_scaler.transform(X_cv)
# X_test  = std_scaler.transform(X_test)

In [50]:
from sklearn.metrics import classification_report

def evaluate(preds, y, target_names, title=""):
    print(f"--- {title} ---")
    print(f"accuracy: {(preds == y).mean():.4f}")
    print(classification_report(y, preds, target_names=target_names))
    
hyper_params = {"n_estimators": 300, "learning_rate": 0.2, "verbosity": 1,
             "random_state": RANDOM_STATE,  "early_stopping_rounds": 20, "tree_method": "hist"}
eval_set = [(X_cv, y_cv)]

xgb_model = XGBClassifier(**hyper_params)
xgb_model.fit(X_train, y_train, eval_set = eval_set, verbose=0)

preds = xgb_model.predict(X_test)
evaluate(preds, y_test, ["benign", "anomaly"], "baseline")

del xgb_model

--- baseline ---
accuracy: 0.7889
              precision    recall  f1-score   support

      benign       0.79      1.00      0.88    805504
     anomaly       0.97      0.03      0.05    222835

    accuracy                           0.79   1028339
   macro avg       0.88      0.51      0.47   1028339
weighted avg       0.83      0.79      0.70   1028339



In [51]:
results = pd.DataFrame({
    "attack": y_test_attackname.values,   # raw string labels you kept
    "pred":   preds,                      # 0/1 from the model
})
# recall per type = fraction of that type's rows flagged as attack (pred==1)
per_type = results[results["attack"] != "BENIGN"].groupby("attack")["pred"].agg(
    recall="mean", n="size"
)
print(per_type.sort_values("recall"))

                             recall       n
attack                                     
Bot                        0.000000    1948
Infiltration               0.000000      36
PortScan                   0.005293   90694
DDoS                       0.028083  128014
Web Attack  Sql Injection  0.619048      21
Web Attack  Brute Force    0.850340    1470
Web Attack  XSS            0.920245     652


day by day split, every attack class in the test set is an unseen type. we only got 3%. web attacks have much better recall than the others, possibly because they resemble the DoS attacks from the wednesday set.

In [52]:
train_attacks

{'DoS GoldenEye',
 'DoS Hulk',
 'DoS Slowhttptest',
 'DoS slowloris',
 'FTP-Patator',
 'Heartbleed',
 'SSH-Patator'}

In [53]:
test_attacks

{'Bot',
 'DDoS',
 'Infiltration',
 'PortScan',
 'Web Attack  Brute Force',
 'Web Attack  Sql Injection',
 'Web Attack  XSS'}